In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
df = pd.read_csv("data.csv")

control = df[df.group=="control"]["conversion"]
treatment = df[df.group=="treatment"]["conversion"]

-H0: conversion_rate_control = conversion_rate_treatment
-H1: conversion_rate_control ≠ conversion_rate_treatment

In [3]:
stat, p = stats.ttest_ind(control, treatment)
stat, p

(np.float64(-0.09131836123684156), np.float64(0.9272579539840706))

p value is > 0.05. Hence we fail to reject the null hypothesis

In [4]:
diff = treatment.mean() - control.mean()

se = np.sqrt(control.var()/len(control) + treatment.var()/len(treatment))

ci_low = diff - 1.96*se
ci_high = diff + 1.96*se

(ci_low, ci_high)

(np.float64(-0.040926744927377384), np.float64(0.04492674492737739))

In [5]:
pooled_std = np.sqrt(
((len(control)-1)*control.std()**2 + (len(treatment)-1)*treatment.std()**2) /
(len(control)+len(treatment)-2)
)

cohens_d = diff/pooled_std

cohens_d

np.float64(0.005775480274049003)

The experiment tested whether the new website design improves the conversion rate compared to the current version.
The statistical test produced a p-value of 0.927, which is much higher than the significance threshold of 0.05.
This means the observed difference in conversion rates could easily occur due to random variation in the data.
Therefore, we cannot conclude that the new design improves conversions.
More data or a larger effect size would be needed to confidently detect a difference.

In [6]:
import numpy as np
from scipy import stats


def ab_test(control, treatment, alpha=0.05):

    control = np.array(control)
    treatment = np.array(treatment)

    # normality test
    p_control = stats.shapiro(control)[1]
    p_treatment = stats.shapiro(treatment)[1]

    normal = p_control > 0.05 and p_treatment > 0.05

    if normal:
        stat, p = stats.ttest_ind(control, treatment)
    else:
        stat, p = stats.mannwhitneyu(control, treatment)

    reject = p < alpha

    diff = treatment.mean() - control.mean()

    pooled_std = np.sqrt(
        ((len(control)-1)*control.std()**2 + (len(treatment)-1)*treatment.std()**2)
        /(len(control)+len(treatment)-2)
    )

    effect = diff / pooled_std

    se = np.sqrt(control.var()/len(control) + treatment.var()/len(treatment))
    ci = (diff - 1.96*se, diff + 1.96*se)

    return {
        "statistic": stat,
        "p_value": p,
        "reject_H0": reject,
        "effect_size": effect,
        "ci_95": ci
    }